## Reading the distillation set silent file

This notebook loads the distillation-set silent file that lives inside the repository tree when the file is available locally.

Because the silent file is large, it is not meant to be version-controlled. The notebook looks for it in:
- `Distillation_set_curation/distillation_set.silent` by default, or
- the path provided by the `DISTILLATION_SILENT_PATH` environment variable.

The code below also adds `TMB_RFD2/TMB_RFD2_code` to `sys.path` so the notebook can import the shared silent-file reader from this repository.

Run this notebook in a Python environment that has PyRosetta available.

In [1]:
from pathlib import Path
import os
import sys


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Distillation_set_curation").is_dir() and (candidate / "TMB_RFD2").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the TMB_pipeline repository root. Run the notebook from inside the cloned repository."
    )


repo_root = find_repo_root()
code_dir = repo_root / "TMB_RFD2" / "TMB_RFD2_code"
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

silent_path = Path(
    os.environ.get(
        "DISTILLATION_SILENT_PATH",
        repo_root / "Distillation_set_curation" / "distillation_set.silent",
    )
).expanduser().resolve()

if not silent_path.is_file():
    raise FileNotFoundError(
        f"Silent file not found at {silent_path}. "
        "Either place distillation_set.silent there or set DISTILLATION_SILENT_PATH."
    )

print(f"Repository root: {repo_root}")
print(f"Silent file: {silent_path}")
print(f"Added to sys.path: {code_dir}")

Repository root: /home/rias04/TMB_pipeline
Silent file: /home/rias04/TMB_pipeline/Distillation_set_curation/distillation_set.silent
Added to sys.path: /home/rias04/TMB_pipeline/TMB_RFD2/TMB_RFD2_code


In [1]:
from rf_diffusion.import_pyrosetta import prepare_pyrosetta, rosetta as ros
from rf_diffusion.silent_files import read_pose_from_silent

prepare_pyrosetta({"pyrosetta_flags": "-mute all -beta_nov16"})

sfd = ros().core.io.silent.SilentFileData(ros().core.io.silent.SilentFileOptions())
sfd.read_file(str(silent_path))
tags = list(sfd.tags())

if not tags:
    raise ValueError(f"No structures were found in {silent_path}.")

print(f"Loaded {len(tags)} structures from the silent file.")
print("First 10 tags:")
for tag in tags[:10]:
    print(f"- {tag}")

example_tag = tags[0]
pose, _ = read_pose_from_silent(str(silent_path), example_tag, sfd=sfd)

print(f"\nExample tag: {example_tag}")
print(f"Example pose length: {pose.total_residue()}")
print(f"Example sequence: {pose.sequence()}")

ModuleNotFoundError: No module named 'rf_diffusion'